# NEUMANN AM2: free Colab native BF16 Gate0

Hardware metadata only. Does not execute AM1, old 12 tasks, or any model.

1. Select a GPU runtime only if the Colab UI confirms free account eligibility.
   No premium resources, purchase, or cycling GPU allocations. Notebook metadata
   does not request a GPU. GPU availability/type is not guaranteed:
   https://research.google.com/colaboratory/faq.html
2. Confirm free runtime using the flag below; otherwise an entitlement blocker
   is saved without a physical probe. Run these three code cells once.
3. Save the receipt ZIP before disconnecting. Stop after qualification.

The unchanged CP1 gate requires accessible CUDA, CC >=8.0, total VRAM >=14 GiB,
and native BF16 (including_emulation=False). T4 stops before any model.
The wrapper adds current free VRAM and CUDA/PyTorch/driver versions. Metadata
queries may initialize a CUDA context but run no tensor arithmetic or fit test.

Even hardware PASS cannot run AM2: no admitted 24-original/3-origin cohort,
trusted generic-original checkers or matched TOOL+/structural bindings yet.
AM1 first FAIL unchanged; Q1-Q7 OPEN; no sealed/frontier access.
No package install, model download, credential or account identifier is requested.


In [ ]:
# Confirm ONLY after checking the free runtime in Colab UI.
FREE_COLAB_RUNTIME_CONFIRMED = False


In [ ]:
# Public source bytes only; no pip or weights.
import hashlib, json, pathlib, subprocess, sys, tempfile, uuid, urllib.request
SOURCE_COMMIT = 'd9665d5520c5954c6dff5bb3c3fcb9605e6f04ff'
PINNED_FILES = {'experiments/am2_colab_hardware_receipt.py': '2d0397499653e292abaea523bed89f2815ba0ea3b137ff98e6f24c1120fb85de', 'experiments/am2_cp1_native_bf16_preflight.py': 'e515443496fd7799c961edeef0eb748ad429b90c5bca57a588260dcfce28dc13', 'docs/experiments/results/am2_colab_preparation_20261019/MODEL_IDENTITY.json': '03836e730bcb2532763fc9f8134c65e8cf2d57e6bc796c6e953cc8754eef536a'}
ROOT = pathlib.Path(tempfile.mkdtemp(prefix="neumann_am2_gate0_"))
for relative, expected in PINNED_FILES.items():
    url = f"https://raw.githubusercontent.com/yoonsj0305/NEUMANN/{SOURCE_COMMIT}/{relative}"
    data = urllib.request.urlopen(url, timeout=30).read()
    if hashlib.sha256(data).hexdigest() != expected:
        raise RuntimeError("SOURCE_HASH_MISMATCH: " + relative)
    target = ROOT / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(data)
identity = json.loads((ROOT / "docs/experiments/results/am2_colab_preparation_20261019/MODEL_IDENTITY.json").read_text())
print("Contract identity only; no weights loaded:", identity)


In [ ]:
# Last cell: hardware metadata only, NEVER model inference.
import shutil
RUN_ID = uuid.uuid4().hex
OUT = ROOT / "receipts" / RUN_ID
command = [sys.executable, str(ROOT / "experiments/am2_colab_hardware_receipt.py"), "--out", str(OUT)]
if FREE_COLAB_RUNTIME_CONFIRMED is True:
    command.append("--confirm-free-colab-runtime")
try:
    process = subprocess.run(command, cwd=ROOT, capture_output=True, text=True, timeout=60)
except subprocess.TimeoutExpired:
    # subprocess.run kills and waits for this metadata-only child. No retry.
    print("BLOCKED_HARDWARE_PROBE_TIMEOUT; no model run, no retry.")
    raise
print(process.stdout)
if process.returncode not in (0, 2):
    raise RuntimeError("HARDWARE_WRAPPER_ERROR exit=" + str(process.returncode))
receipt = json.loads((OUT / "GPU_QUALIFICATION.json").read_text())
print(json.dumps(receipt, indent=2))
archive = shutil.make_archive(str(ROOT / ("AM2_GATE0_" + RUN_ID)), "zip", OUT)
print("SAVE BEFORE DISCONNECT:", archive)
print("AM2 BLOCKED / NOT_EVALUATED: missing cohort and generic arm/checker bindings.")
print("No model loading, generation, training or frontier call has run.")
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Outside Colab: download the ZIP from the printed location.")
